# 05 — Hierarchical Bayes M3 (⭐⭐ PRIMARY INFERENTIAL)

**Notebook id**: `05_hierarchical_bayes` → vault `03.6-modele-hierarchique.md`

The refactored **heterogeneous** hierarchical model, fitted under **three competing pooling structures** so the topographic organisation is an empirical **RESULT tested by LOO**, not an assumed hypothesis (revue-methodo 2026-05-29):

- **Likelihood**: `pm.Bernoulli`/logit for the binary compartments **PTE/PTI/CFE/CFI** (0 events ≥ 2); `pm.OrderedLogistic` on the collapsed **{0, 1, ≥2}** scale (2 cutpoints) for **trochlée, rotule**.
- **Free cutpoints per compartment** — cutpoints encode *measurement* (baseline prevalence), never pooled.
- **Pooling on the Group×Time slopes** `delta_comp`, with the grouping set by the structure under test:
  - `exchangeable` → one knee-wide mean ⇒ exposes **`delta_bar`** (the selection-immune global estimand);
  - `two_block` → {PF}/{FT} ⇒ exposes `delta_pf`, `delta_ft`, `contrast_pf_ft` (a **candidate** structure);
  - `three_cluster` → {PF}/{FT-antlat}/{FT-med}.
- η = β_c[comp]·t + γ·g + δ_comp[comp]·t·g + u[patient], **t ∈ {−0.5,+0.5}** the **S2 − S1 contrast** (not a per-year slope); Student-t(3) priors; patient random intercept `u` retained.

**Primary global estimand** = **δ̄** (`delta_bar`) from the *exchangeable* fit: a pre-specified directional H1 ("cyclops worsen cartilage"), so it uses the one-sided rule `P(δ̄>0|data) ≥ 0.95` (`two_sided=False`). It is reported **honestly**: the knee-wide average **dilutes** a compartment-specific signal, so it is expected to be *small and uncertain* — we do **not** overclaim it.

**Topography** = a RESULT: `bm.compare_pooling_structures` ranks the three fits by **LOO** (ELPD ± SE). At n=69 this is likely only *indicative* (small ΔELPD with a comparable dse) — reported, not over-claimed.

**PF localisation** = read **non-circularly** from `bm.derived_pf_contrast` on the *neutral exchangeable* posterior (no PF/FT partition imposed). The two-block `contrast_pf_ft` is shown as a **candidate-structure** output and judged by the **two-sided** credible rule (post-hoc ⇒ HDI excludes 0), never the one-sided rule.

**NUTS**: `chains=4, tune=2000, draws=2000, target_accept=0.95, seed=42`, sampled with **nutpie** (the host has no C compiler for the default PyTensor backend). **Convergence**: R̂ ≤ 1.01, ESS_bulk ≥ 400, divergences = 0; `fit_m3_with_escalation` auto-escalates (target_accept=0.99, tune=4000) once if needed.

In [ ]:
import sys
from pathlib import Path

current = Path().absolute().parent
sys.path.insert(0, (current / "src").as_posix())


In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED, SITES, SITES_PF, SITES_FT, SITES_BINARY, SITES_ORDINAL,
    GROUPS, BLOCKS, N_TOTAL, N_TOTAL_ANALYSABLE, N_MENISCUS, N_CYCLOPS,
    SCORE_MAX, SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)      # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)            # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)             # one row per patient (group, anonyme)
patient = pp.to_patient(df)       # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [c for c in ['group','anonyme','female','sexe','pivot_pivot_contact',
                    'travail_physique','tabac','age_at_trauma','imc','taille','poids']
        if c in patient.columns]
merged = wide.merge(patient[_cov], on=['group','anonyme'], how='left')

print('long:', df.shape, '| wide:', wide.shape, '| patient:', patient.shape,
      '| merged:', merged.shape)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(['group','anonyme']).size() == 2).all(), 'composite key broken'


## 1. Model data shape (long-long, collapsed {0,1,≥2})

One row per (patient × time × compartment), NaN scores dropped, grade 3 → 2. The two likelihood blocks (binary vs ordinal) are split inside the model builder.

In [ ]:
long_long = bm._melt_long_long(df, SITES, 'anonyme', 'group', collapse=True)
print('long_long shape:', long_long.shape, '(non-NaN site cells, collapsed)')
print('y categories:', sorted(long_long['y'].unique()), '(<=2 => collapse applied)')
print()
print('Block per compartment (0=PF, 1=FT):')
print(long_long.groupby('comp')['block_idx'].first())
print('\nBinary compartments (Bernoulli):',
      long_long.loc[long_long.is_binary, 'comp'].unique().tolist())
print('Time contrast values (S1=-0.5, S2=+0.5):', sorted(long_long['t'].unique()))


## 2. Fit & compare the three pooling structures by LOO (runnable — sampling)

This cell **is runnable** (uncommented) and re-runs NUTS via nutpie **three times** (exchangeable, two_block, three_cluster), each with the pointwise log-likelihood, then ranks them by `arviz.compare` (ELPD-LOO). It returns `{idatas, loos, compare}`. **This is the test of whether the data reject 6-site exchangeability in favour of a block structure** — the topography is thereby an empirical result, not an assumption.

> At n=69 the comparison is likely only **indicative** (small ΔELPD, comparable `dse`). Read the ELPD differences *and their SE*; do not over-claim a winner.

In [ ]:
# Fit all three pooling structures and rank by LOO. Runs NUTS x3 via nutpie.
res = bm.compare_pooling_structures(df, sites=SITES)
idatas = res['idatas']
idata_exch = idatas['exchangeable']      # neutral / selection-immune (PRIMARY)
idata_2blk = idatas['two_block']         # candidate topographic structure
print('LOO comparison (ranked by ELPD-LOO; topography tested, not assumed):')
print(res['compare'])
# Optional: cache the neutral posterior for make_figures.py / reuse.
# idata_exch.to_netcdf('../results/idata_m3_exchangeable.nc')


## 3. Convergence diagnostics on every structure (R̂ ≤ 1.01, ESS ≥ 400, div = 0)

All three fits must converge before the LOO ranking is trustworthy.

In [ ]:
diags = {k: bm.check_convergence(v, rhat_max=1.01, ess_min=400)
         for k, v in idatas.items()}
for k, d in diags.items():
    print(f"  {k:14s} ok={d['ok']}  max_rhat={d['max_rhat']:.3f}  "
          f"min_ess_bulk={d['min_ess_bulk']:.0f}  n_div={d['n_divergent']}")
diag = diags['exchangeable']            # primary structure's diagnostics
assert all(d['ok'] for d in diags.values()), \
    f"convergence failed: {[(k, d['offenders'][:4]) for k, d in diags.items() if not d['ok']]}"


## 4. PRIMARY global verdict — knee-wide δ̄ (exchangeable, pre-specified)

δ̄ = `delta_bar` is the **selection-immune global** Group×Time interaction from the *exchangeable* model (no PF/FT partition imposed). H1 ("cyclops worsen cartilage") was pre-specified and **directional**, so we use the legitimate **one-sided** rule `P(δ̄ > 0 | data) ≥ 0.95` (`two_sided=False, direction='greater'`).

Reported **honestly**: because the knee-wide average **dilutes** a compartment-specific effect, δ̄ is expected to be *small and uncertain* — a wide HDI straddling 0 is the honest answer, **not** an overclaimed global effect.

In [ ]:
print('Exchangeable posterior summary (delta_bar = knee-wide global):')
print(rpt.summary_bayes(idata_exch, var_names=['delta_bar','gamma','sigma_delta','sigma_u'],
                        hdi_prob=0.94))
print()
v_global = rpt.verdict_bayes_en(
    idata_exch, 'delta_bar', 'Primary global knee-wide effect (delta-bar)',
    threshold=0.95, two_sided=False, direction='greater')   # pre-specified directional
print(v_global['sentence'])
print(rpt.confidence_phrase_en(
    v_global['p_direction'], 'the knee-wide cartilage worsens more in cyclops'))
print()
print('NOTE: delta_bar is the honest PRIMARY estimand. A small / 0-straddling HDI'
      ' reflects DILUTION of a compartment-specific signal, not absence of an effect'
      ' in the patellofemoral block (localised below).')


## 5. PF localisation — `derived_pf_contrast` on the NEUTRAL posterior (non-circular)

The honest, **non-circular** reading of where the signal sits: take the joint posterior of the per-compartment `delta_comp` from the *exchangeable* fit (which imposed **no** PF/FT partition) and form (mean δ over PF sites) − (mean δ over FT sites). This reveals the pattern the data show without *assuming* the partition that the contrast measures.

In [ ]:
pf = bm.derived_pf_contrast(idata_exch)
print('Derived PF - FT contrast (from the neutral exchangeable delta_comp):')
print(f"  PF sites      : {pf['pf_sites']}")
print(f"  FT sites      : {pf['ft_sites']}")
print(f"  mean          : {pf['mean']:+.3f}")
print(f"  {int(pf['hdi_prob']*100)}% HDI       : [{pf['hdi_lo']:+.3f}, {pf['hdi_hi']:+.3f}]")
print(f"  P(PF-FT > 0)  : {pf['p_gt0']:.3f}")
print('  => the worsening is localised to the patellofemoral block (data-revealed,')
print('     not assumed); the partition was selected, so the amplitude is an')
print('     optimistic bound.')


## 6. CANDIDATE structure — two-block `contrast_pf_ft` (post-hoc, two-sided rule)

The `two_block` fit is one **candidate** topographic structure (the one LOO compares above). Its deterministic `contrast_pf_ft` = δ_PF − δ_FT is a **post-hoc** estimand (the PF/FT split was suggested by the data), so it is judged by the direction-agnostic **two-sided** credible rule (HDI excludes 0) — **never** the one-sided P>0 rule, which would double-count the data.

In [ ]:
print('Two-block posterior summary (candidate structure):')
print(rpt.summary_bayes(idata_2blk,
                        var_names=['delta_pf','delta_ft','contrast_pf_ft','gamma'],
                        hdi_prob=0.94))
print()
v_contrast = rpt.verdict_bayes_en(
    idata_2blk, 'contrast_pf_ft',
    'Topographic specificity PF-FT (candidate two-block structure)',
    two_sided=True)                                         # post-hoc => two-sided
v_pf_blk = rpt.verdict_bayes_en(
    idata_2blk, 'delta_pf', 'PF block delta (candidate structure)', two_sided=True)
print(v_pf_blk['sentence'])
print(v_contrast['sentence'])
print('  (labelled CANDIDATE: this structure is one of three compared by LOO above.)')


## 7. Forest plot — two-block posterior effects (94% HDI)

Visualises the candidate two-block estimands; the global δ̄ verdict lives in section 4 (exchangeable).

In [ ]:
fig = viz.forest_m3(idata_2blk,
                    var_names=('delta_pf','delta_ft','contrast_pf_ft','gamma'))
fig


## 8. Convergence diagnostic plots (trace + rank) — exchangeable (primary)

In [ ]:
fig = viz.diagnostics_m3(idata_exch,
                         var_names=('delta_bar','gamma'),
                         convergence=diag)
fig


## 9. Per-compartment slopes & free cutpoints (exchangeable, interpretation)

β_c is the per-compartment time slope; the cutpoints are free per ordinal compartment (measurement, not pooled).

In [ ]:
import arviz as az
print('Per-compartment slopes beta_c (94% HDI, exchangeable):')
print(rpt.summary_bayes(idata_exch, var_names=['beta_c'], hdi_prob=0.94))
print()
cs = bm.cutpoint_summary(idata_exch)
print('Free cutpoints per ordinal compartment:')
for comp, d in cs.get('per_comp', {}).items():
    print(f"  {comp:10s} cutpoints ~ {[round(x,2) for x in d['cut_means']]}")


## 10. Posterior predictive check (optional, ~30 s after the fit)

Category frequencies {0, 1, ≥2} per (group, time, compartment): empirical vs posterior-predictive draws, across both likelihood blocks.

In [ ]:
# Optional PPC on the exchangeable fit (uncomment to run after the fits).
# ppc = bm.ppc_m3(idata_exch, df, n_draws=500)
# print('Observed (head):'); print(ppc['observed_freq'].head(10))
# print('Predicted mean (head):'); print(ppc['predicted_freq_mean'].head(10))
print('PPC cell ready (commented). Uncomment to run after fitting M3.')


## 11. Prior sensitivity on σ_β (optional, exploratory)

Re-fit with `sigma_beta` HalfNormal scale in (0.5, 1.0, 5.0) — tight / default / loose pooling — and compare the posterior of `delta_pf` and `contrast_pf_ft` (two-block). Each refit re-runs NUTS; run only if you want the grid.

In [ ]:
# Optional prior sensitivity grid (each line re-runs NUTS via nutpie).
# from constants import SENSITIVITY_PRIOR_SIGMAS
# rows = []
# for s in SENSITIVITY_PRIOR_SIGMAS:
#     id_s = bm.fit_m3_with_prior(df, prior_sigma_beta=s)   # two_block by default
#     for v in ['delta_pf','contrast_pf_ft']:
#         vals = id_s.posterior[v].values.ravel()
#         rows.append(dict(prior_sigma_beta=s, var=v, mean=float(vals.mean()),
#                          p_gt0=float((vals>0).mean())))
# print(pd.DataFrame(rows))
print('Prior-sensitivity grid ready (commented). Uncomment to run 3 refits.')


## Sanity asserts (post-fit)

All three structures converged; the primary verdict is on δ̄ (pre-specified one-sided). The PF localisation / candidate contrast are **not** asserted (post-hoc, honestly reported).

In [ ]:
assert all(d['ok'] for d in diags.values()), 'a pooling structure failed to converge'
assert diag['max_rhat'] <= 1.01 and diag['min_ess_bulk'] >= 400 and diag['n_divergent'] == 0
assert set(res['idatas']) == {'exchangeable','two_block','three_cluster'}
assert 'delta_bar' in idata_exch.posterior          # primary estimand exposed
assert 'contrast_pf_ft' in idata_2blk.posterior     # candidate-structure estimand
print('M3 convergence + structure-comparison asserts passed.')
print('Primary global verdict (delta_bar):', v_global['supported'])
